In [11]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

In [12]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=api_key)

In [13]:
def ask_llm(prompt):

    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_completion_tokens=1500
    )

    return completion.choices[0].message.content

In [14]:
def create_plan(task):

    prompt = f"""
You are an AI planning agent.

Break the task into steps.

Return ONLY JSON.

Example format:

{{
 "task": "{task}",
 "steps": [
   "step1",
   "step2",
   "step3"
 ]
}}

Task:
{task}
"""

    response = ask_llm(prompt)

    return json.loads(response)

In [15]:
def save_file(filename, content):

    with open(filename, "w") as f:
        f.write(content)

    return "File saved"


def read_file(filename):

    with open(filename, "r") as f:
        return f.read()


tools = {
    "save_file": save_file,
    "read_file": read_file
}


In [ ]:
tool_description = """
Available tools:

save_file
input: filename, content

read_file
input: filename
"""

In [17]:
def decide_tool(step):

    prompt = f"""
You are an AI agent.

{tool_description}

Choose the best tool for the step.

Return ONLY JSON.

Example:

{{
 "tool": "tool_name",
 "input": {{
   "param": "value"
 }}
}}

Step:
{step}
"""

    response = ask_llm(prompt)

    return json.loads(response)

In [19]:

def run_agent(plan):


    print("\nPLAN:")
    for step in plan["steps"]:
        print("-", step)

    for step in plan["steps"]:

        print("\nExecuting step:", step)

        action = decide_tool(step)

        tool_name = action["tool"]
        tool_input = action["input"]

        result = tools[tool_name](**tool_input)

        print("Result:", result)

In [20]:
plan = create_plan("Create a file called hello.txt with text 'Hello Agentic AI'")


In [21]:
run_agent(plan)


PLAN:
- Open a text editor or use a command-line interface.
- Create a new file named hello.txt.
- Enter the text: Hello Agentic AI
- Save the file.
- Close the editor or exit the command-line session.
- Optionally, verify the file contents by opening hello.txt or using a command like `cat hello.txt`.

Executing step: Open a text editor or use a command-line interface.
Result: File saved

Executing step: Create a new file named hello.txt.
Result: File saved

Executing step: Enter the text: Hello Agentic AI
Result: File saved

Executing step: Save the file.
Result: File saved

Executing step: Close the editor or exit the command-line session.
Result: File saved

Executing step: Optionally, verify the file contents by opening hello.txt or using a command like `cat hello.txt`.
Result: 
